# ML model preparation (YOLO)

<div style="text-align: justify">

Notebook used to train and validate an object detection model, especially a YOLOv5 model. Model is trained on the exported LARD datasets from [`data-export.ipynb`](./data-export.ipynb) notebook.
</div>

> Notebook inspired from G. Delhomme's work. [[Github]](https://github.com/geoffrey-g-delhomme/lard-yolov8)

## General helpers

In [2]:
from pathlib import Path
from typing import (
    Union,
    Tuple,
    List
)

import os, sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import cv2
import shutil
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import torch
from ultralytics import YOLO

from swmf.data import create_yolo_datayaml

<div style='text-align: justify', class="alert alert-danger">

Make sure to have the correct path to exported LARD dataset, from the previous [`data-export.ipynb`](./data-export.ipynb) notebook.
</div>

In [5]:
PATH_TO_EXPORTED_LARD = "../data/datasets/lard_512x512_ICPR2026"

In [6]:
### Reproductibility ###
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
########################

In [7]:
### Training params  ###
D_IMGSZ = [512, 512]  # [W, H]
D_BATCH = 64
D_N_EPOCHS = 20
D_MODEL_NAME = "yolov5n.pt"  # Use smallest versions for RT (yolov5n, yolov8n, ...)
D_MODEL_TASK = "detect"
########################

In [8]:
def YOLO_train(
        dataset_path: Path,
        model_name: Union[str, Path] = D_MODEL_NAME,
        imgsz: Union[int, Tuple[int,int]] = D_IMGSZ,
        batch: int = -1,
        epochs: int = D_N_EPOCHS,
        resume: bool = False,
        **kwargs,
): 
    """
    Train a YOLO model on given dataset.

    Args:
        dataset_path (Path): the path to the train dataset
        model_name (Union[str, Path]): the name/path of/to the model
        imgsz (Union[int, Tuple[int, int]]): the size of the images
        batch (int): the batch size
        epochs (int): the number of training epochs
        resume (bool): whether to resume from past checkpoint or start from scratch

    Returns:
        (ultralytics.DetMetrics) The training result.
    """
    data_path = dataset_path / 'data.yaml'
    devices = ','.join([str(i) for i in range(torch.cuda.device_count())])
    batches = torch.cuda.device_count() * batch

    result = YOLO(model_name).train(
        data=data_path,
        imgsz=imgsz,
        batch=batches,
        device=devices,
        epochs=epochs,
        resume=resume,
        **kwargs,
    )
    return result


def YOLO_valid(
        dataset_path: Path,
        model: YOLO,
): 
    """
    Compute metrics for trained model on the validation dataset.
    
    Args:
        dataset_path (Path): The path to the dataset
        model (YOLO): The trained YOLO model

    Returns:
        The metrics for validation.
    """
    result = model.val(data=dataset_path / 'data.yaml')
    return result


def YOLO_test(
        dataset_path: Path,
        model: YOLO,
):
    """
    Compute metrics for trained model on the test dataset.

    Args:
        dataset_path (Path): The path to the dataset
        model (YOLO): The trained YOLO model

    Returns:
        The metrics for test.
    """
    result = model.val(data=dataset_path / 'data_test.yaml')
    return result


def YOLO_predict(
        sources_path: Union[Path, List[Path]],
        model: YOLO,
        **kwargs,
):
    """
    Launch YOLO prediction on given data.

    Args:
        sources_path (Union[Path, List[Path]]): The path to the input data.
        model (YOLO): The trained YOLO model.

    Returns:
        The results of the YOLO prediction routine.
    """
    devices = ','.join([str(i) for i in range(torch.cuda.device_count())])
    result = model.predict(
        source=sources_path,
        device=devices,
        save=True,
        save_txt=True,
        show_labels=True,
        verbose=False,
        **kwargs,
    )
    return result


def YOLO_export(
        model: YOLO,
        export_format: str = "onnx",
):
    """
    Export a YOLO model to any supported format
    """
    model.export(format=export_format, dynamic=True, simplify=True)


In [9]:
def _util_cp_training_result_data(src: Path, dst: Path):
    """
    Copy content of 'src' folder into 'dst'. 

    Args:
        src (Path): The path to Ultralytics 'save_dir'
        dst (Path): The path to your model 'save_dir'

    Note:
        The function saves the model weights, the training args and the generated plots.
    """
    if not src.exists():
        raise ValueError(f"Source path does not exist. {src.as_posix()}")

    dst.mkdir(parents=True, exist_ok=True)
    sub = dst / "training_info"
    sub.mkdir(parents=True, exist_ok=True)

    for f in src.iterdir():
        if f.is_file() and f.suffix in ['.yaml', '.png']:
            shutil.copy2(f, sub)
            print(f"✅ Copied {f.name} to {sub.as_posix()}")
    
    for f in (src / "weights").iterdir():
        if f.is_file() and f.suffix in [".pt", ".onnx"]:
            shutil.copy2(f, dst)
            print(f"✅ Copied {f.name} to {dst.as_posix()}")


def _util_rm_yolo_training_folder(root: Path = Path("runs")):
    """
    Remove the YOLO 'runs/' directory.

    Args:
        root (Path): The path to base runs/ folder.
    """
    if not root.exists():
        print(f"Nothing to delete. {root} does not exist.")
        return
    
    shutil.rmtree(root)
    print("✅ Deleted YOLO 'runs/' directory.")
    

<div style="text-align: justify">

Launch YOLO training below.
</div>

In [10]:
lard_data_dpath = Path(PATH_TO_EXPORTED_LARD).resolve()
SPLITS = [
    "split_trainval", 
    "split_trainval_per_runway", 
    "split_trainval_per_airport"
]

### EDIT HERE ###
SPLIT_INDX = 1  # BY DEFAULT WE KEEP THE SPLIT BY RUNWAYS
#################

split_fpath = lard_data_dpath / f"{SPLITS[SPLIT_INDX]}.csv"
split_dname = SPLITS[SPLIT_INDX]

## YOLO training

In [19]:
# Define path to yolo training data (data.yaml for split!)
yolo_data_dpath = create_yolo_datayaml(
    dataset_dpath=lard_data_dpath,
    yolo_task=D_MODEL_TASK,
    split_fpath=split_fpath,
    split_dname=split_dname,
)
yolo_data_dpath

PosixPath('/home/dariom/Workspace/LARD_monitoring/data/datasets/lard_512x512_ICPR2026/task_detect/split_trainval_per_runway')

In [20]:
# Define path to yolo training save directory
yolo_save_dpath = Path("../data/models").resolve() / D_MODEL_TASK / D_MODEL_NAME.rpartition('.')[0] / (lard_data_dpath.stem + "_" + split_fpath.stem) / f"{D_N_EPOCHS:03d}_epochs"
yolo_save_dpath.as_posix()

'/home/dariom/Workspace/LARD_monitoring/data/models/detect/yolov5n/lard_512x512_ICPR2026_split_trainval_per_runway/020_epochs'

In [21]:
### EDIT HERE ###
TRAIN_STATE = 0  # {0: train from scratch, 1: train from chckpnt, 2: no train}
#################

In [22]:
# Launch YOLO training according to command state
match TRAIN_STATE:
    case 0:
        result = YOLO_train(
            dataset_path=yolo_data_dpath,
            model_name=D_MODEL_NAME,
            imgsz=D_IMGSZ,
            batch=-1,
            epochs=D_N_EPOCHS,
            # patience=3,
            # resume=False,
        )
    case 1:
        result = YOLO_train(
            dataset_path=yolo_data_dpath,
            model_name=yolo_save_dpath / "last.pt",
            # imgsz=D_IMGSZ,
            # batch=D_BATCH,
            epochs=D_N_EPOCHS,
            resume=True,
        )
    case _:
        print("No training required.")

# Delete YOLO runs/ folder if necessary
if TRAIN_STATE <= 1:
    _util_cp_training_result_data(result.save_dir, yolo_save_dpath)
    _util_rm_yolo_training_folder(Path("../runs/").resolve())

PRO TIP 💡 Replace 'model=yolov5n.pt' with new 'model=yolov5nu.pt'.
YOLOv5 'u' models are trained with https://github.com/ultralytics/ultralytics and feature improved performance vs standard YOLOv5 models trained with https://github.com/ultralytics/yolov5.



New https://pypi.org/project/ultralytics/8.3.236 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.220 🚀 Python-3.10.18 torch-2.9.0+cu128 CUDA:0 (NVIDIA RTX A1000 6GB Laptop GPU, 6144MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/dariom/Workspace/LARD_monitoring/data/datasets/lard_512x512_ICPR2026/task_detect/split_trainval_per_runway/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=[512, 512], int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0

### Evaluate the model

In [23]:
model = YOLO(yolo_save_dpath / "best.pt")
model.fuse()
model.info(verbose=True)

YOLOv5n summary (fused): 84 layers, 2,503,139 parameters, 0 gradients, 7.1 GFLOPs
YOLOv5n summary (fused): 84 layers, 2,503,139 parameters, 0 gradients, 7.1 GFLOPs


(84, 2503139, 0, 7.0648832)

In [24]:
metrics_valid = YOLO_valid(yolo_data_dpath, model)

print("=== [Valid] metrics ===")
print(metrics_valid)

Ultralytics 8.3.220 🚀 Python-3.10.18 torch-2.9.0+cu128 CUDA:0 (NVIDIA RTX A1000 6GB Laptop GPU, 6144MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1693.8±577.6 MB/s, size: 96.8 KB)
val: Scanning /home/dariom/Workspace/LARD_monitoring/data/datasets/lard_512x512_ICPR2026/task_detect/labels/valid.cache... 1458 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1458/1458 698.3Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 92/92 5.1it/s 18.1s0.2s
                   all       1458       1458      0.962      0.949      0.982      0.698
Speed: 0.8ms preprocess, 6.6ms inference, 0.0ms loss, 1.4ms postprocess per image
Results saved to /home/dariom/Workspace/LARD_monitoring/runs/detect/val
=== [Valid] metrics ===
ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0

In [25]:
metrics_tests = YOLO_test(yolo_data_dpath, model)

print("=== [Tests] metrics ===")
print(metrics_tests)

Ultralytics 8.3.220 🚀 Python-3.10.18 torch-2.9.0+cu128 CUDA:0 (NVIDIA RTX A1000 6GB Laptop GPU, 6144MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 151.5±44.0 MB/s, size: 108.1 KB)
val: Scanning /home/dariom/Workspace/LARD_monitoring/data/datasets/lard_512x512_ICPR2026/task_detect/labels/test.cache... 2212 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2212/2212 2.6Mit/s 0.0s0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 139/139 7.8it/s 17.7s0.1s
                   all       2212       2212      0.929       0.92      0.957      0.672
Speed: 0.4ms preprocess, 3.6ms inference, 0.0ms loss, 1.0ms postprocess per image
Results saved to /home/dariom/Workspace/LARD_monitoring/runs/detect/val2
=== [Tests] metrics ===
ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 

Save the evaluation results into `./results` directory

In [26]:
def _util_cp_eval_result_data(src: Path, dst: Path):
    """
    """
    assert src.exists(), f"Src path does not exist... {src.as_posix()}"

    dst.mkdir(parents=True, exist_ok=True)
    for f in src.iterdir():
        if (src/f).is_file():
            shutil.copy2(src/f, dst)

In [27]:
path_to_results = Path('../results').resolve()
path_to_results = path_to_results / PATH_TO_EXPORTED_LARD.split('/')[-1] / D_MODEL_TASK / D_MODEL_NAME.rpartition('.')[0] / (lard_data_dpath.stem + "_" + split_fpath.stem) / f"{D_N_EPOCHS:03d}_epochs"
path_to_results.as_posix()

'/home/dariom/Workspace/LARD_monitoring/results/lard_512x512_ICPR2026/detect/yolov5n/lard_512x512_ICPR2026_split_trainval_per_runway/020_epochs'

In [28]:
# Copy valid results
valid_dst_dpath = path_to_results / split_dname / 'valid'
valid_src_dpath = Path(metrics_valid.save_dir).resolve()

_util_cp_eval_result_data(valid_src_dpath, valid_dst_dpath)

In [29]:
# Copy tests results
tests_dst_dpath = path_to_results / split_dname / 'test'
tests_src_dpath = Path(metrics_tests.save_dir).resolve()

_util_cp_eval_result_data(tests_src_dpath, tests_dst_dpath)

In [30]:
# Clean ./runs yolo directory
_util_rm_yolo_training_folder(Path("../runs/").resolve())

✅ Deleted YOLO 'runs/' directory.


Export the YOLO model to ONNX

In [31]:
YOLO_export(model)

Ultralytics 8.3.220 🚀 Python-3.10.18 torch-2.9.0+cu128 CPU (13th Gen Intel Core i7-13850HX)

PyTorch: starting from '/home/dariom/Workspace/LARD_monitoring/data/models/detect/yolov5n/lard_512x512_ICPR2026_split_trainval_per_runway/020_epochs/best.pt' with input shape (1, 3, 512, 512) BCHW and output shape(s) (1, 5, 5376) (5.0 MB)

ONNX: starting export with onnx 1.19.1 opset 22...
ONNX: slimming with onnxslim 0.1.71...


2025-12-11 16:50:07.364144254 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


ONNX: export success ✅ 2.2s, saved as '/home/dariom/Workspace/LARD_monitoring/data/models/detect/yolov5n/lard_512x512_ICPR2026_split_trainval_per_runway/020_epochs/best.onnx' (9.9 MB)

Export complete (2.3s)
Results saved to /home/dariom/Workspace/LARD_monitoring/data/models/detect/yolov5n/lard_512x512_ICPR2026_split_trainval_per_runway/020_epochs
Predict:         yolo predict task=detect model=/home/dariom/Workspace/LARD_monitoring/data/models/detect/yolov5n/lard_512x512_ICPR2026_split_trainval_per_runway/020_epochs/best.onnx imgsz=512  
Validate:        yolo val task=detect model=/home/dariom/Workspace/LARD_monitoring/data/models/detect/yolov5n/lard_512x512_ICPR2026_split_trainval_per_runway/020_epochs/best.onnx imgsz=512 data=/home/dariom/Workspace/LARD_monitoring/data/datasets/lard_512x512_ICPR2026/task_detect/split_trainval_per_runway/data.yaml  
Visualize:       https://netron.app


## YOLO prediction

In [32]:
def draw_bbox(lab_filepath: Path, img_filepath: Path = None, ax = None, is_quiet=False, *plt_args, **plt_kwargs):
    """
    """
    sample = pd.read_csv(lab_filepath.as_posix(), delimiter=' ', header=None)
    bbox = sample.iloc[0].to_numpy()[1:]

    if img_filepath is None:
        img_filepath = lab_filepath.parent.parent.parent / "images" / lab_filepath.parent.stem / f"{lab_filepath.stem}.jpg"
    
    image = np.array(cv2.cvtColor(cv2.imread(img_filepath.as_posix()), cv2.COLOR_BGR2RGB))
    h,w,d = image.shape
    bbox[0] *= w
    bbox[1] *= h
    bbox[2] *= w
    bbox[3] *= h

    bbox_xyxy = [
        bbox[0] - bbox[2]/2., 
        bbox[1] - bbox[3]/2., 
        bbox[0] + bbox[2]/2., 
        bbox[1] + bbox[3]/2., 
    ]
    image = cv2.rectangle(image, 
                          (int(bbox_xyxy[0]), int(bbox_xyxy[1])),
                          (int(bbox_xyxy[2]), int(bbox_xyxy[3])),
                          color=(255, 0, 0), thickness=2)
    
    if not is_quiet:
        if ax is None:
            fig, ax = plt.subplots(1, 1, *plt_args, **plt_kwargs)
            ax.imshow(image)
            ax.axis('off')
            fig.tight_layout()
        else:
            ax.imshow(image)
            ax.axis('off')
    return image

In [33]:
### Reproducibility ###
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
#######################

In [34]:
path_to_test_images = lard_data_dpath / "images" / "test"
path_to_test_images

PosixPath('/home/dariom/Workspace/LARD_monitoring/data/datasets/lard_512x512_ICPR2026/images/test')

In [35]:
n_samples = 10

test_imgs_relpath = np.random.choice([f for f in os.listdir(path_to_test_images)], size=n_samples)
test_imgs_abspath = [path_to_test_images / f for f in test_imgs_relpath]
results = YOLO_predict(test_imgs_abspath, model, imgsz=D_IMGSZ, max_det=1)

Results saved to /home/dariom/Workspace/LARD_monitoring/runs/detect/predict
9 labels saved to /home/dariom/Workspace/LARD_monitoring/runs/detect/predict/labels


In [36]:
# Copy tests results
predict_dst_dpath = path_to_results / split_dname / 'predict'
predict_src_dpath = Path(results[0].save_dir).resolve()

_util_cp_eval_result_data(
    predict_src_dpath, 
    predict_dst_dpath
)

_util_cp_eval_result_data(
    predict_src_dpath / 'labels', 
    predict_dst_dpath / 'labels'
)

In [37]:
# Delete /runs directory
_util_rm_yolo_training_folder(Path("../runs/").resolve())

✅ Deleted YOLO 'runs/' directory.


## Exploration

*(independent section)*

In [114]:
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

In [115]:
import torch

In [116]:
# Define path to yolo training save directory
yolo_path = Path("../data/models").resolve() / D_MODEL_TASK / D_MODEL_NAME.rpartition('.')[0] / (lard_data_dpath.stem + "_" + split_fpath.stem) / f"010_epochs"
yolo_path.as_posix()

'/home/dariom/Workspace/LARD_monitoring/data/models/detect/yolov5n/lard_512x512_ICPR2026_split_trainval_per_runway/010_epochs'

In [117]:
yolov5_detector = YOLO(yolo_path / "best.pt")
yolov5_detector.fuse()
yolov5_detector.info(verbose=True)
yolov5_detector.eval();

YOLOv5n summary (fused): 84 layers, 2,503,139 parameters, 0 gradients, 7.1 GFLOPs
YOLOv5n summary (fused): 84 layers, 2,503,139 parameters, 0 gradients, 7.1 GFLOPs


In [118]:
yolov5_detector.model.model[-1]

Detect(
  (cv2): ModuleList(
    (0): Sequential(
      (0): Conv(
        (conv): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (act): SiLU(inplace=True)
      )
      (2): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1))
    )
    (1): Sequential(
      (0): Conv(
        (conv): Conv2d(128, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (act): SiLU(inplace=True)
      )
      (2): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1))
    )
    (2): Sequential(
      (0): Conv(
        (conv): Conv2d(256, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(64, 64, kernel_size=(3, 3),

In [119]:
test_imgs_path = Path('../data/datasets/lard_512x512_ICPR2026/images/test')
test_imgs = [
    test_imgs_path / "000001.jpg",
    test_imgs_path / "000002.jpg",
    test_imgs_path / "000428.jpg",
]

def logits_hook(module, i_data, o_data):
    global logits
    logits = o_data

hook = yolov5_detector.model.model[-1].register_forward_hook(logits_hook)

with torch.no_grad():
    result = yolov5_detector.predict(test_imgs, imgsz=512)

hook.remove()


0: 512x512 1 runway, 43.0ms
1: 512x512 1 runway, 43.0ms
2: 512x512 1 runway, 43.0ms
Speed: 5.4ms preprocess, 43.0ms inference, 2.7ms postprocess per image at shape (1, 3, 512, 512)


In [120]:
print(len(result))
print(result[0].boxes.data[0,:4])
print(result[1].boxes.data[0,:4])

3
tensor([296.7838, 257.0732, 313.2778, 278.9243], device='cuda:0')
tensor([133.0459, 295.0984, 146.0947, 318.6529], device='cuda:0')


In [121]:
print(len(logits))
print(logits[0].shape)
print(len(logits[1]))
print(logits[1][0].shape)
print(logits[1][1].shape)
print(logits[1][2].shape)
# print(logits[1][2][0])

2
torch.Size([3, 5, 5376])
3
torch.Size([3, 65, 64, 64])
torch.Size([3, 65, 32, 32])
torch.Size([3, 65, 16, 16])


In [122]:
torch.cat([l.view(*l.shape[:2], -1) for l in logits[1]], dim=2).shape

torch.Size([3, 65, 5376])

In [123]:
ll = logits[0].detach().cpu().numpy()

sorted_idx = np.argsort(-ll[:,4,:], axis=1)
sorted_log = np.zeros_like(ll)
for b in range(ll.shape[0]):
    sorted_log[b,:,:] = ll[b,:, sorted_idx[b,:]].T
sorted_log[:,:,0]

array([[     296.78,      257.07,      313.28,      278.92,      0.8591],
       [     133.05,       295.1,      146.09,      318.65,     0.83019],
       [     162.25,      231.26,      226.51,      281.76,      0.9033]], dtype=float32)

In [124]:
# 1. Extract raw bboxes (logits[0])
raw_bboxes = logits[0][:,:4,:].permute(0, 2, 1).detach().cpu().numpy()
raw_logits = torch.cat([l.view(*l.shape[:2], -1) for l in logits[1]], dim=2).detach().cpu().numpy()

# 2. Find matching bbox in final preds
match_indice = []
for b in range(len(result)):
    final_bboxes = result[b].boxes.data[:,:4].detach().cpu().numpy()
    batch_indice = []

    for final_bbox in final_bboxes:
        d = np.sqrt(np.sum((raw_bboxes[b] - final_bbox)**2, axis=1))
        batch_indice.append(np.argmin(d))
    
    match_indice.append(np.array(batch_indice))

# 3. Extract correspondig logits
final_logits = []
for b in range(len(result)):
    batch_logits = raw_logits[b, :, match_indice[b]]  # 65, n
    final_logits.append(batch_logits)

final_logits = np.concatenate(final_logits, 0)

In [125]:
final_logits

array([[     9.8066,      9.4038,      3.0844,     0.40908,     -1.0389,     -2.3053,     -3.6801,     -4.0756,     -3.5208,     -4.3225,     -4.5588,     -4.6608,     -4.9396,      -5.151,     -5.2614,     -4.7968,      9.0141,      8.4571,      2.0981,    -0.16014,    0.073488,     -1.7002,     -3.6255,
            -4.6421,     -5.3499,     -5.5362,     -5.8158,     -6.0281,     -6.2073,       -6.34,     -6.3778,     -5.6515,      3.4213,      8.7128,      9.3732,      3.6616,     0.13977,     -1.7061,     -3.3661,     -4.0385,     -3.9324,     -4.1376,      -4.079,     -4.0219,      -4.155,      -4.435,
             -3.987,     -3.2852,    -0.49104,      2.1282,      7.6105,      7.0661,      1.3139,     -1.5825,     -2.4814,     -3.4738,     -4.6312,     -4.7348,     -4.5037,     -4.5452,     -4.8148,     -4.9925,     -5.1215,     -4.6148,      1.8078],
       [     7.9703,      9.7563,      5.3944,     0.86449,   0.0084256,     -2.9214,     -4.5272,     -4.4399,     -3.6519,     -